# GEMS TCO Missing Data Analysis — 2022/2023/2024/2025 July

직접 pkl 파일을 열어 엄밀하게 검증합니다 (`load_maxmin_ordered_data_bymonthyear` 없이).

도메인: lat [-3, 2], lon [121, 131]  
분석 내용:
- 연도별 전체 통계 비교 (관측수, 미싱수, 비율)
- 일별 미싱% (4년 나란히)
- 슬롯별 미싱% heatmap (Day × Hour)
- 공간 분포 (격자별 미싱 빈도)
- 데이터 이상 탐지 (날짜 누락, 슬롯 수 이상)

**검증된 사실**: 2022-07-29 누락(30일치만 존재), 2025-07-24 슬롯 7개(나머지 8개)

In [ ]:
import pickle, sys, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
YEARS = ['2022', '2023', '2024', '2025']

DATA_ROOT = Path('/Users/joonwonlee/Documents/GEMS_DATA')
OUT_DIR   = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {'2022': '#e41a1c', '2023': '#ff7f00', '2024': '#377eb8', '2025': '#4daf4a'}

print('Data root:', DATA_ROOT)
print('Output  :', OUT_DIR)

In [ ]:
# ── 직접 pkl 로드 (loader 통하지 않음) ───────────────────────────────

def load_year_july(year: str) -> dict:
    yy = year[2:]
    path = DATA_ROOT / f'pickle_{year}' / f'tco_grid_{yy}_07.pkl'
    with open(path, 'rb') as f:
        raw = pickle.load(f)
    print(f'{year}: loaded {len(raw)} slots from {path.name}')
    return raw

def apply_bbox(df):
    mask = ((df['Latitude']  >= LAT_RANGE[0]) & (df['Latitude']  <= LAT_RANGE[1]) &
            (df['Longitude'] >= LON_RANGE[0]) & (df['Longitude'] <= LON_RANGE[1]))
    return df[mask].reset_index(drop=True)

def parse_key(k):
    # raw key format: 'y24m07day01_hm00:53'
    m_day = re.search(r'day(\d+)', k)
    m_hm  = re.search(r'hm(\d+):(\d+)', k)
    day    = int(m_day.group(1)) if m_day else -1
    hour   = int(m_hm.group(1))  if m_hm  else -1
    minute = int(m_hm.group(2))  if m_hm  else 0
    return day, hour, minute

raws = {}
for yr in YEARS:
    try:
        raws[yr] = load_year_july(yr)
    except FileNotFoundError as e:
        print(f'  SKIP {yr}: {e}')

print('\nYears available:', list(raws.keys()))

In [ ]:
# ── 데이터 검증 및 이상 탐지 ──────────────────────────────────────────

for yr, raw in raws.items():
    keys  = sorted(raw.keys())
    days  = sorted({parse_key(k)[0] for k in keys})
    slots_per_day = {d: sum(1 for k in keys if parse_key(k)[0] == d) for d in days}
    odd_days = {d: n for d, n in slots_per_day.items() if n != 8}

    # 첫 슬롯 bbox 체크
    k0   = keys[0]
    df0  = apply_bbox(raw[k0])
    n_cells = len(df0)

    print(f'\n=== {yr}-07 ===')
    print(f'  Total slots : {len(keys)}')
    print(f'  Days present: {len(days)} ({min(days)}..{max(days)})')

    missing_days = [d for d in range(1, 32) if d not in days]
    if missing_days:
        print(f'  *** MISSING DAYS: {missing_days} ***')
    if odd_days:
        print(f'  *** DAYS WITH ≠8 SLOTS: {odd_days} ***')

    print(f'  Grid cells (after bbox): {n_cells:,}')
    print(f'  Lat: {df0.Latitude.min():.4f}..{df0.Latitude.max():.4f}  '
          f'Lon: {df0.Longitude.min():.4f}..{df0.Longitude.max():.4f}')
    print(f'  Raw rows: {len(raw[k0])} → bbox filtered: {n_cells} '
          f'(dropped {len(raw[k0])-n_cells})')

In [ ]:
# ── 전체 슬롯별 미싱 통계 계산 ────────────────────────────────────────

all_slot_df = {}   # year -> DataFrame
all_daily   = {}   # year -> DataFrame

for yr, raw in raws.items():
    keys = sorted(raw.keys())
    # bbox filter 기준 격자 수 (모든 슬롯에서 동일하다고 가정, 검증)
    n_cells = len(apply_bbox(raw[keys[0]]))

    records = []
    for k in keys:
        dff = apply_bbox(raw[k])
        if len(dff) != n_cells:
            print(f'WARNING {yr} {k}: {len(dff)} rows (expected {n_cells})')
        n_total = len(dff)
        n_miss  = int(dff['ColumnAmountO3'].isna().sum())
        day, hour, minute = parse_key(k)
        records.append({
            'key': k, 'day': day, 'hour': hour, 'minute': minute,
            'n_total': n_total, 'n_miss': n_miss,
            'pct_miss': n_miss / n_total * 100 if n_total > 0 else 0,
        })

    df_s = pd.DataFrame(records).sort_values(['day','hour','minute']).reset_index(drop=True)
    df_s['slot_idx'] = df_s.groupby('day').cumcount()

    daily = df_s.groupby('day').agg(
        n_slots   = ('key','count'),
        n_total   = ('n_total','sum'),
        n_miss    = ('n_miss','sum'),
        pct_max   = ('pct_miss','max'),
        pct_min   = ('pct_miss','min'),
        pct_mean  = ('pct_miss','mean'),
    ).reset_index()
    daily['pct_total'] = daily.n_miss / daily.n_total * 100

    all_slot_df[yr] = df_s
    all_daily[yr]   = daily

    tot_obs  = df_s.n_total.sum()
    tot_miss = df_s.n_miss.sum()
    print(f'{yr}: {len(keys)} slots | {tot_obs:,} obs | '
          f'{tot_miss:,} miss ({tot_miss/tot_obs*100:.2f}%) | '
          f'median slot={df_s.pct_miss.median():.2f}% | '
          f'max slot={df_s.pct_miss.max():.2f}%')

In [ ]:
# ── Figure 1: 연도별 전체 통계 비교 (bar chart) ───────────────────────
years_avail = list(raws.keys())
overall_pcts = []
med_slot     = []
max_slot     = []
for yr in years_avail:
    df_s = all_slot_df[yr]
    tot_obs  = df_s.n_total.sum()
    tot_miss = df_s.n_miss.sum()
    overall_pcts.append(tot_miss / tot_obs * 100)
    med_slot.append(df_s.pct_miss.median())
    max_slot.append(df_s.pct_miss.max())

x = np.arange(len(years_avail))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w, overall_pcts, w, label='Overall missing %',    color='steelblue', alpha=0.85)
b2 = ax.bar(x,     med_slot,     w, label='Median slot missing %', color='darkorange', alpha=0.85)
b3 = ax.bar(x + w, max_slot,     w, label='Max slot missing %',    color='tomato', alpha=0.85)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}%',
                ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([f'{yr}-07' for yr in years_avail], fontsize=11)
ax.set_ylabel('Missing %')
ax.set_title('GEMS TCO Missing Data — July Comparison (lat[-3,2], lon[121,131])\n'
             'Direct pkl read, bbox filtered, no ordering applied')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(max_slot) * 1.2)
plt.tight_layout()
plt.savefig(OUT_DIR / 'miss4yr_overall_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── Figure 2: 일별 미싱% 4년 나란히 (2×2 grid) ───────────────────────

fig, axes = plt.subplots(2, 2, figsize=(20, 10), sharey=False)
axes = axes.flatten()

for i, yr in enumerate(years_avail):
    ax  = axes[i]
    dly = all_daily[yr]
    df_s = all_slot_df[yr]

    c = COLORS[yr]
    ax.bar(dly['day'], dly['pct_total'], color=c, alpha=0.7, label='일별 총 miss%', zorder=2)
    ax.errorbar(
        dly['day'], dly['pct_mean'],
        yerr=[dly['pct_mean'] - dly['pct_min'], dly['pct_max'] - dly['pct_mean']],
        fmt='o', color='black', ms=3, lw=1, capsize=3, zorder=3,
        label='slot mean ± [min,max]'
    )

    tot_obs  = df_s.n_total.sum()
    tot_miss = df_s.n_miss.sum()
    n_days   = len(dly)
    n_keys   = len(df_s)

    ax.set_title(
        f'{yr}-07  |  {n_days} days, {n_keys} slots\n'
        f'Overall miss: {tot_miss/tot_obs*100:.2f}%  '
        f'(median slot: {df_s.pct_miss.median():.2f}%  max: {df_s.pct_miss.max():.2f}%)',
        fontsize=10
    )
    ax.set_xlabel('Day of July')
    ax.set_ylabel('Missing %')
    ax.set_xticks(range(1, 32, 2))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    # Annotate missing days
    for d in range(1, 32):
        if d not in dly['day'].values:
            ax.axvline(d, color='grey', lw=0.8, linestyle=':', alpha=0.6)
            ax.text(d, ax.get_ylim()[1] * 0.95, 'miss', ha='center',
                    fontsize=6, color='grey', rotation=90)

fig.suptitle('Daily Missing Rate by Year — GEMS TCO July  (direct pkl, bbox only)',
             fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'miss4yr_daily_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── Figure 3: Day×Slot heatmap (4년) ──────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(22, 14))
axes = axes.flatten()

for i, yr in enumerate(years_avail):
    ax   = axes[i]
    df_s = all_slot_df[yr]

    pivot = df_s.pivot_table(index='day', columns='slot_idx', values='pct_miss', aggfunc='first')

    # hour:minute labels
    slot_labels = [
        f"{int(df_s[df_s.slot_idx==si]['hour'].median()):02d}:"
        f"{int(df_s[df_s.slot_idx==si]['minute'].median()):02d}"
        for si in sorted(df_s.slot_idx.unique())
    ]

    vmax = min(pivot.values[~np.isnan(pivot.values)].max(), 40)
    im = ax.imshow(
        pivot.values, aspect='auto', cmap='YlOrRd',
        vmin=0, vmax=vmax, interpolation='nearest'
    )
    plt.colorbar(im, ax=ax, label='Missing %', shrink=0.8)

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'D{d}' for d in pivot.index], fontsize=7)
    ax.set_xticks(range(len(slot_labels)))
    ax.set_xticklabels(slot_labels, fontsize=8, rotation=40, ha='right')
    ax.set_xlabel('Slot (UTC)')
    ax.set_ylabel('Day')

    # Annotate cells >15%
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            v = pivot.values[r, c]
            if not np.isnan(v) and v > 15:
                ax.text(c, r, f'{v:.0f}', ha='center', va='center',
                        fontsize=6, color='black', fontweight='bold')

    ax.set_title(f'{yr}-07  |  missing % per slot (>15 annotated)', fontsize=11)

fig.suptitle('Day × Time-Slot Missing% Heatmap — GEMS TCO July\nlat[-3,2], lon[121,131]',
             fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'miss4yr_heatmap_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── Figure 4: 시간대별 평균 미싱% (4년 오버레이) ──────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

for yr in years_avail:
    df_s = all_slot_df[yr]
    # 슬롯 인덱스(0~7)별 집계
    hourly = df_s.groupby('slot_idx').agg(
        pct_mean = ('pct_miss','mean'),
        pct_std  = ('pct_miss','std'),
        label    = ('hour', lambda x: f"{int(x.median()):02d}:{0:02d}"),
    ).reset_index()

    ax.plot(hourly['slot_idx'], hourly['pct_mean'], 'o-',
            color=COLORS[yr], lw=2, ms=6, label=yr)
    ax.fill_between(
        hourly['slot_idx'],
        hourly['pct_mean'] - hourly['pct_std'],
        hourly['pct_mean'] + hourly['pct_std'],
        color=COLORS[yr], alpha=0.15
    )

# x-tick labels from 2024
df24 = all_slot_df['2024']
slot_labels = [
    f"{int(df24[df24.slot_idx==si]['hour'].median()):02d}:"
    f"{int(df24[df24.slot_idx==si]['minute'].median()):02d}"
    for si in sorted(df24.slot_idx.unique())
]
ax.set_xticks(range(len(slot_labels)))
ax.set_xticklabels(slot_labels, fontsize=9)
ax.set_xlabel('Time slot (UTC)')
ax.set_ylabel('Mean missing % (± 1σ over all July days)')
ax.set_title('Intra-day missing pattern — 4-year overlay (July)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'miss4yr_hourly_overlay_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── 공간 분석: 격자별 미싱 빈도 (4년 합산 및 연도별) ──────────────────

# 공통 격자 좌표 (2024 기준)
ref_key = sorted(raws['2024'].keys())[0]
ref_df  = apply_bbox(raws['2024'][ref_key]).reset_index(drop=True)
n_cells = len(ref_df)
lats    = ref_df['Latitude'].values
lons    = ref_df['Longitude'].values

# 격자 구조
unique_lats = np.sort(np.unique(np.round(lats, 4)))[::-1]
unique_lons = np.sort(np.unique(np.round(lons, 4)))
lat_to_row  = {la: i for i, la in enumerate(unique_lats)}
lon_to_col  = {lo: i for i, lo in enumerate(unique_lons)}
n_lat, n_lon = len(unique_lats), len(unique_lons)
print(f'Grid: {n_lat} lat × {n_lon} lon = {n_lat*n_lon:,} cells ({n_cells:,} in domain)')

# 연도별 공간 미싱 빈도
miss_frac_yr = {}
for yr, raw in raws.items():
    keys = sorted(raw.keys())
    mc = np.zeros(n_cells, dtype=np.int32)
    tc = np.zeros(n_cells, dtype=np.int32)
    for k in keys:
        dff = apply_bbox(raw[k]).reset_index(drop=True)
        if len(dff) != n_cells:
            continue
        mc += dff['ColumnAmountO3'].isna().values.astype(np.int32)
        tc += 1
    miss_frac_yr[yr] = np.where(tc > 0, mc / tc, np.nan)
    print(f'{yr}: mean miss frac = {np.nanmean(miss_frac_yr[yr]):.4f}  '
          f'max = {np.nanmax(miss_frac_yr[yr]):.4f}  '
          f'always-miss cells = {int((miss_frac_yr[yr]==1).sum())}  '
          f'never-miss cells = {int((miss_frac_yr[yr]==0).sum())}')

# 4년 합산
fracs_stack = np.stack([miss_frac_yr[yr] for yr in years_avail], axis=0)
miss_frac_all = np.nanmean(fracs_stack, axis=0)

In [ ]:
# ── Figure 5: 공간 미싱 분포 (연도별 grid imshow) ────────────────────

def to_grid(miss_frac):
    g = np.full((n_lat, n_lon), np.nan)
    for i, (la, lo) in enumerate(zip(lats, lons)):
        r = lat_to_row.get(round(la, 4))
        c = lon_to_col.get(round(lo, 4))
        if r is not None and c is not None:
            g[r, c] = miss_frac[i] * 100
    return g

grids = {yr: to_grid(miss_frac_yr[yr]) for yr in years_avail}
grid_all = to_grid(miss_frac_all)
extent = [unique_lons[0], unique_lons[-1], unique_lats[-1], unique_lats[0]]

fig, axes = plt.subplots(2, 3, figsize=(22, 10))
axes = axes.flatten()

for i, yr in enumerate(years_avail):
    im = axes[i].imshow(grids[yr], aspect='auto', cmap='YlOrRd',
                         vmin=0, vmax=50, extent=extent, interpolation='nearest')
    plt.colorbar(im, ax=axes[i], label='Miss %')
    axes[i].set_title(f'{yr}-07\nmean={np.nanmean(grids[yr]):.2f}%', fontsize=10)
    axes[i].set_xlabel('Lon'); axes[i].set_ylabel('Lat')

# 4년 평균
im5 = axes[4].imshow(grid_all, aspect='auto', cmap='YlOrRd',
                      vmin=0, vmax=50, extent=extent, interpolation='nearest')
plt.colorbar(im5, ax=axes[4], label='Miss %')
axes[4].set_title('4-year mean (2022-2025)\n(July average)', fontsize=10)
axes[4].set_xlabel('Lon'); axes[4].set_ylabel('Lat')

# 연도간 표준편차
grid_std = np.nanstd(np.stack(list(grids.values()), axis=0), axis=0)
im5b = axes[5].imshow(grid_std, aspect='auto', cmap='Blues',
                       vmin=0, extent=extent, interpolation='nearest')
plt.colorbar(im5b, ax=axes[5], label='Std dev %')
axes[5].set_title('Inter-year std dev\n(year-to-year variability)', fontsize=10)
axes[5].set_xlabel('Lon'); axes[5].set_ylabel('Lat')

fig.suptitle('Spatial missing% per grid cell — GEMS TCO July  (lat[-3,2], lon[121,131])',
             fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'miss4yr_spatial_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── Figure 6: 히스토그램 비교 (격자별 미싱 분율) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) per-year overlaid histograms
bins = np.linspace(0, 1, 51)
for yr in years_avail:
    axes[0].hist(miss_frac_yr[yr], bins=bins, histtype='step', lw=2,
                 color=COLORS[yr], label=f'{yr} (mean={np.nanmean(miss_frac_yr[yr]):.3f})')
axes[0].set_xlabel('Per-cell missing fraction (all July slots)')
axes[0].set_ylabel('Number of grid cells')
axes[0].set_title('Distribution of per-cell missing fraction by year')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_yscale('log')

# (b) 4년 평균 히스토그램
axes[1].hist(miss_frac_all, bins=bins, color='purple', alpha=0.7,
             edgecolor='white', lw=0.3)
axes[1].axvline(np.nanmean(miss_frac_all), color='tomato', lw=2,
                linestyle='--', label=f'mean={np.nanmean(miss_frac_all):.3f}')
axes[1].set_xlabel('Per-cell missing fraction (4-year average)')
axes[1].set_ylabel('Number of grid cells')
axes[1].set_title('4-year averaged per-cell missing fraction')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'miss4yr_hist_050626.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# ── 최종 요약 테이블 ──────────────────────────────────────────────────
print('=' * 70)
print('GEMS TCO MISSING DATA SUMMARY — July (direct pkl, bbox only)')
print(f'Domain: lat {LAT_RANGE}, lon {LON_RANGE}')
print(f'Grid: {n_lat} lat × {n_lon} lon = {n_cells:,} cells per slot')
print('=' * 70)

rows = []
for yr in years_avail:
    df_s    = all_slot_df[yr]
    tot_obs = df_s.n_total.sum()
    tot_miss= df_s.n_miss.sum()
    mf      = miss_frac_yr[yr]
    rows.append({
        'Year': yr,
        'N_slots': len(df_s),
        'N_days' : df_s.day.nunique(),
        'Total_obs': tot_obs,
        'Total_miss': tot_miss,
        'Overall%': round(tot_miss/tot_obs*100, 2),
        'Median_slot%': round(df_s.pct_miss.median(), 2),
        'p90_slot%': round(np.percentile(df_s.pct_miss, 90), 2),
        'Max_slot%': round(df_s.pct_miss.max(), 2),
        'Always-miss_cells': int((mf == 1.0).sum()),
        'Never-miss_cells':  int((mf == 0.0).sum()),
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))

df_summary.to_csv(OUT_DIR / 'miss4yr_summary_050626.csv', index=False)
print('\nCSV saved.')

print()
print('=== DATA ANOMALIES ===')
for yr, raw in raws.items():
    keys = sorted(raw.keys())
    days_present = sorted({parse_key(k)[0] for k in keys})
    missing_days = [d for d in range(1, 32) if d not in days_present]
    slots_per_day = pd.Series({d: sum(1 for k in keys if parse_key(k)[0] == d)
                               for d in days_present})
    odd = slots_per_day[slots_per_day != 8]
    if missing_days or len(odd) > 0:
        print(f'  {yr}: missing days={missing_days}, odd-slot days={odd.to_dict()}')
    else:
        print(f'  {yr}: OK (31 days, 248 slots)')